[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-genai/parallelization-agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# Google GenAI SDK - 构建并行智能体

## 安装依赖

In [ ]:
!pip install -q google-genai arize-phoenix-otel openinference-instrumentation-google-genai

## 连接 Arize Phoenix

In [ ]:
import os
from getpass import getpass

from google import genai
from phoenix.otel import register

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

tracer_provider = register(auto_instrument=True, project_name="google-genai-parallelization-agent")
tracer = tracer_provider.get_tracer(__name__)

## 通过 Google Vertex AI 进行身份验证

In [ ]:
!gcloud auth login

In [ ]:
# 使用 Vertex AI API 创建客户端，此处也可以改用 Google GenAI API
client = genai.Client(vertexai=True, project="<填写 GCP 项目 ID>", location="us-central1")

# 并行执行

In [ ]:
import concurrent.futures
import textwrap

from opentelemetry.context import attach, detach, get_current

GEMINI_MODEL_NAME = "gemini-2.0-flash-001"


def define_research_prompts():
    """定义不同主题的研究提示词。"""
    return {
        "ai_research_result": (
            "Artificial Intelligence",
            """你是 AI 研究助手。
研究‘人工智能’的最新进展。
简要总结关键发现（1–2 句）。
重点使用知识截止日期之前已知的信息。
仅输出摘要。""",
        ),
        "quantum_research_result": (
            "Quantum Computing",
            """你是专注物理与计算的 AI 研究助手。
研究‘量子计算’的最新突破。
简要总结关键发现（1–2 句）。
重点使用知识截止日期之前已知的信息。
仅输出摘要。""",
        ),
        "biotech_research_result": (
            "Biotechnology",
            """你是专注生命科学的 AI 研究助手。
研究‘生物技术’的最新创新。
简要总结关键发现（1–2 句）。
重点使用知识截止日期之前已知的信息。
仅输出摘要。""",
        ),
    }


@tracer.chain()
def run_research_task(topic, prompt):
    """调用生成模型并返回文本结果。"""
    print(f"开始研究： {topic}...")
    try:
        response = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=prompt)
        print(f"已完成研究： {topic}.")
        return response.text.strip()
    except Exception as e:
        print(f"研究以下主题时出错： {topic}: {e}")
        return f"获取以下主题信息时出错： {topic}."


# 此辅助函数为 run_research_task 函数封装 OTel 上下文。
# 若不这样做，每个研究子任务会作为新跨度追踪，无法嵌套
# 在主研究任务的跨度之下。
def context_wrapped_task(context, func, *args, **kwargs):
    token = attach(context)
    try:
        return func(*args, **kwargs)
    finally:
        detach(token)


@tracer.chain()
def execute_parallel_research(research_prompts):
    """并行执行研究任务并返回结果。"""
    research_results = {}
    context = get_current()  # 捕获当前 OTel 上下文
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # 提交任务，并为每个任务封装上下文
        future_to_key = {
            executor.submit(context_wrapped_task, context, run_research_task, topic, prompt): key
            for key, (topic, prompt) in research_prompts.items()
        }
        # 在各任务完成时收集结果
        for future in concurrent.futures.as_completed(future_to_key):
            key = future_to_key[future]
            try:
                result = future.result()
                research_results[key] = result
            except Exception as exc:
                print(f"{key} 产生异常： {exc}")
                research_results[key] = f"发生错误： {key} task."
    return research_results


@tracer.chain()
def create_synthesis_prompt(research_results):
    """创建用于综合研究结果的提示词。"""
    return f"""你是负责将研究发现整合为结构化报告的 AI 助手。

主要任务是综合下列研究摘要，并明确说明各项发现所属领域（AI、量子计算、生物技术）。
为每个主题设置标题，确保报告连贯，自然整合关键要点。

**关键要求：整个回复必须仅以如下‘输入摘要’中的信息为依据。
不得添加这些摘要中没有的外部知识、事实或细节。**

**输入摘要：**

*   **AI 进展：** {research_results.get("ai_research_result", "N/A")}
*   **量子计算：** {research_results.get("quantum_research_result", "N/A")}
*   **Biotechnology:** {research_results.get("biotech_research_result", "N/A")}

生成最终综合报告。
"""


@tracer.chain()
def synthesize_results(synthesis_instruction):
    """根据研究结果生成综合报告。"""
    print("\n--- 开始综合 ---")
    try:
        synthesis_response = client.models.generate_content(
            model=GEMINI_MODEL_NAME, contents=synthesis_instruction
        )
        final_report = synthesis_response.text.strip()
        print("--- 综合完成 ---")
        return final_report
    except Exception as e:
        print(f"综合时出错： {e}")
        return "生成最终报告时出错。"


def display_report(final_report):
    """显示最终综合报告。"""
    print("\n=== 最终综合报告 ===\n")
    # 使用 textwrap 处理 Notebook 中可能较长的报告
    print(textwrap.fill(final_report, width=80))


# 主执行流程
@tracer.agent()
def main():
    research_prompts = define_research_prompts()
    research_results = execute_parallel_research(research_prompts)

    print("\n--- 并行研究结果 ---")
    for key, result in research_results.items():
        print(f"{key}: {result}")

    synthesis_instruction = create_synthesis_prompt(research_results)
    final_report = synthesize_results(synthesis_instruction)
    display_report(final_report)
    return final_report


main()